### Start of GOALS

### 1. Data Scraping

In [2]:
"""
GOALS Project - Full FBref Scraper
====================================
Scrapes all stat types for Premier League, La Liga, Bundesliga
across 3 seasons and saves as JSON files.

BEFORE RUNNING:
1. Open https://fbref.com in Edge
2. F12 → Network → refresh → click first request → Headers
3. Copy the 'cookie:' value and paste it into COOKIES below

pip install requests beautifulsoup4 lxml pandas
"""

import cloudscraper
from bs4 import BeautifulSoup, Comment
import pandas as pd
import time
import os

OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# ── PASTE YOUR FRESH COOKIES HERE ─────────────────────────────
COOKIES = """srcssfull=yes; osano_consentmanager_uuid=ff6a2741-0a02-45f2-83ea-25e14ed5efd7; _ga=GA1.1.1005045719.1772577371; _pubcid=2c0ed4af-b1a1-49f6-8eab-98b2b5767519; _lc2_fpi=6fbc725567fc--01kjtxgdg68bz744rw8m32trqc; _lc2_fpi_meta=%7B%22w%22%3A1772577371654%7D; _lr_env_src_ats=false; ccuid=35c0d1ae-1b86-45ce-a99e-728d6077fe58; _cc_id=8e237d4706893798d6d890d956e93d; panoramaId_expiry=1773182171226; panoramaId=d98f5d55b0188f692de1e4385b42185ca02cc17e2b3366a813f584f959934c8b; panoramaIdType=panoDevice; gamera_user_id=9a74a339-f25f-4e0f-8955-8f8f13f0d534; 33acrossIdTp=EmhkvnIsuWQVgaDUvXh%2BIvoP0GG8ZTn6qhOIIYIDn4w%3D; _au_1d=AU1D-0100-001772577372-5LV5026I-S4P0; hubspotutk=6e1f4c43a8363a074b00b98d13271f7a; is_live=true; _li_dcdm_c=.fbref.com; _lr_retry_request=true; __gads=ID=840e5941a11adf81:T=1772577372:RT=1772583219:S=ALNI_MZvPNXQOEXcb94kVXHpHsOcWjc44Q; __eoi=ID=0188330c55d6ea67:T=1772577372:RT=1772583219:S=AA-AfjbbUKoF_808OymoFLiQ4-VO; __cf_bm=__hpOAno4ndM7WwH6t_2M0tZ7cQBrlzTu1aIY1ZOGzg-1772583230-1.0.1.1-NpWlRDSjVzojJQSdqumfLzZRYAaWjD3sWFwYUaLGOiJ.F7xSj4u6wUWMozD1UtaIkv9GqPk8AK2e4r3jU5d1Dgkc.6itCC8YTQUQL10Xnzg; cf_clearance=ECgpZGQCu6tGeYyURLfrDXfx5n2wd9DjOOD8NBl7C.4-1772583235-1.2.1.1-3v6ZTHMK6qrim.W.KAj11z7lOHoCoYiuju2YvuGQl5g6StnRGHLGlSyT2vo0jdq10BfBCXhNaTTrbVvB8PIFDLdfsgmcv6yw31P7ljf2jo.xLFo.YGJZclCW64L0R6hI4DY_RlIX6pjfNulKE8oX5045K11bRQoZU1migqsi0ingCFiC7cG6Ss.7YTgf7r199DCX5Fnxh7IO_qMP8l1lob_r8qq6M785wVm.uB2oqw8; __hstc=218152582.6e1f4c43a8363a074b00b98d13271f7a.1772577395099.1772580491206.1772583253475.3; __hssrc=1; sr_n=1%7CWed%2C%2004%20Mar%202026%2000%3A14%3A29%20GMT; _ga_80FRT7VJ60=GS2.1.s1772583219$o3$g1$t1772583321$j48$l0$h0; _ga_T897NZ0GWZ=GS2.1.s1772583219$o3$g1$t1772583322$j47$l0$h0; __hssc=218152582.5.1772583253475; cto_bundle=63S-_V90dmZMVlE4UHY4VlNBJTJCSUJVS1VCJTJCWXpMNTNjNWlWc216eHJLdmQyYlpFblpMWXZsdElpWnVnZ2RXV09DSTFUZmtEOGM3Zk1zNHdSTUtjN2VPRklTNGZFbjZ6TUpKZEhrcFRYcnhXRlo3QVY3RnZPdlhDdlpodFAxOFg3Q2UlMkJFb2JTQm85QiUyQnRFNWdqRVJzZlFqdnJzQSUzRCUzRA; cto_bidid=6rJLuF91VnZ2dTdiMmVsaXpxUEFuaTJLV0Rhc0xZVHNXN0NraTZNT0slMkZKSlkyVTdWWklEMUJxRTFzYTN0aTElMkJkRlpaSG1Lck5sdWZKRjZHY3hXZ1FST3h5SW5WdzZ2T3JVcjhxMyUyRnR5bWRiOXJJOCUzRA; cto_dna_bundle=63rzpl90dmZMVlE4UHY4VlNBJTJCSUJVS1VCJTJCZHdCQXNKSVBpcnRnY3lTQnklMkJ2ZXR6WU9kZXlCQUVubGhKTDJEZjJSZTQxeDBzOGVJWWVRM01YY0xzcDZXZTJNQSUzRCUzRA"""

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36 Edg/145.0.0.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9,en-GB;q=0.8",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Referer": "https://fbref.com/",
    "Sec-Ch-Ua": '"Not:A-Brand";v="99", "Microsoft Edge";v="145", "Chromium";v="145"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1",
}

# ── Create cloudscraper session with cookies ──────────────────
scraper = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "desktop": True}
)
scraper.headers.update(HEADERS)

# Parse cookie string into session cookies
for cookie in COOKIES.split("; "):
    if "=" in cookie:
        name, value = cookie.split("=", 1)
        scraper.cookies.set(name.strip(), value.strip(), domain=".fbref.com")


In [4]:
# ── Leagues, seasons, stat types ──────────────────────────────
LEAGUES = {
    "premier_league": {"comp_id": "9", "name": "Premier-League"},
    "la_liga": {"comp_id": "12", "name": "La-Liga"},
    "bundesliga": {"comp_id": "20", "name": "Bundesliga"},
}

SEASONS = ["2021-2022", "2022-2023", "2023-2024", "2024-2025"]

STAT_TYPES = {
    "standard":           "stats",
    "shooting":           "shooting",
    "passing":            "passing",
    "passing_types":      "passing_types",
    "goal_shot_creation": "gca",
    "defense":            "defense",
    "possession":         "possession",
    "playing_time":       "playingtime",
    "misc":               "misc",
}

In [ ]:
def get_url(league_key, season, stat_type):
    """Build FBref URL for player stats (note /players/ in path)."""
    league = LEAGUES[league_key]
    cid = league["comp_id"]
    lname = league["name"]
    slug = STAT_TYPES[stat_type]
    return f"https://fbref.com/en/comps/{cid}/{season}/{slug}/players/{season}-{lname}-Stats"


def scrape_table(url):
    """Fetch page and extract the PLAYER stats table (2nd table, skip team stats)."""
    print(f"    Fetching: {url}")
    try:
        resp = scraper.get(url, timeout=30)
    except Exception as e:
        print(f"    ✗ Request error: {e}")
        return None

    print(f"    Status: {resp.status_code}")

    if resp.status_code == 403:
        print("    ✗ BLOCKED — cookies may have expired! Refresh them.")
        return None
    if resp.status_code != 200:
        print(f"    ✗ Error: {resp.status_code}")
        return None

    soup = BeautifulSoup(resp.text, "lxml")

    # Collect ALL tables — both visible and hidden in comments
    all_tables = soup.find_all("table")

    # Also check inside HTML comments (FBref hides tables there)
    for comment in soup.find_all(string=lambda t: isinstance(t, Comment)):
        if "<table" in str(comment):
            comment_soup = BeautifulSoup(str(comment), "lxml")
            for t in comment_soup.find_all("table"):
                all_tables.append(t)

    print(f"    Found {len(all_tables)} tables on page")

    if len(all_tables) == 0:
        print("    ✗ No tables found!")
        return None

    # The FIRST table is team stats (Squad, # Pl, Age, Poss...)
    # The SECOND table is player stats (Rk, Player, Nation, Pos...)
    # We want the one with a "Player" column — find it by checking headers
    player_table = None
    for table in all_tables:
        header_text = table.get_text()[:200]
        if "Player" in header_text and "Rk" in header_text:
            player_table = table
            break

    if player_table is None:
        # Fallback: just grab the second table if available
        if len(all_tables) >= 2:
            player_table = all_tables[1]
            print("    Using 2nd table (fallback)")
        else:
            print("    ✗ Could not find player stats table!")
            return None
    else:
        print("    ✓ Found player stats table")

    df = pd.read_html(str(player_table), header=[0, 1])[0]

    # Flatten multi-level columns
    df.columns = [
        f"{a}_{b}" if "Unnamed" not in str(a) and a != b else str(b)
        for a, b in df.columns
    ]

    # Remove repeated header rows embedded in data
    if "Rk" in df.columns:
        df = df[df["Rk"] != "Rk"].reset_index(drop=True)

    # Drop the "Matches" column if it exists (it's just a link)
    if "Matches" in df.columns:
        df = df.drop(columns=["Matches"])

    print(f"    ✓ {df.shape[0]} players x {df.shape[1]} columns")
    return df

In [6]:
# ══════════════════════════════════════════════════════════════
# TEST SCRAPE — Verify player data before full run
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST: Scraping Premier League 2024-2025 Standard Stats")
print("=" * 60)

test_url = get_url("premier_league", "2024-2025", "standard")
test_df = scrape_table(test_url)

RUN_FULL_SCRAPE = False

if test_df is not None:
    print(f"\n✓ SUCCESS! Got {test_df.shape[0]} players x {test_df.shape[1]} columns")
    print(f"\nFirst 5 players:")
    # Show key columns if they exist
    show_cols = [c for c in ["Player", "Squad", "Pos", "MP", "Gls", "Ast"] if c in test_df.columns]
    if show_cols:
        print(test_df[show_cols].head())
    else:
        print(test_df.head())
    print(f"\nAll columns:\n{list(test_df.columns)}")
    RUN_FULL_SCRAPE = True
else:
    print("\n✗ TEST FAILED")
    print("  1. Go to https://fbref.com in Edge")
    print("  2. F12 → Network → refresh → click first request → Headers")
    print("  3. Copy the full 'cookie:' value")
    print("  4. Replace the COOKIES string at the top and re-run")

TEST: Scraping Premier League 2024-2025 Standard Stats
    Fetching: https://fbref.com/en/comps/9/2024-2025/stats/players/2024-2025-Premier-League-Stats
    Status: 200
    ✓ 20 players x 21 columns

✓ SUCCESS! Got 20 players x 21 columns

First 5 players:
         Squad
0      Arsenal
1  Aston Villa
2  Bournemouth
3    Brentford
4     Brighton

All columns:
['Squad', '# Pl', 'Age', 'Poss', 'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s', 'Performance_Gls', 'Performance_Ast', 'Performance_G+A', 'Performance_G-PK', 'Performance_PK', 'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR', 'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A', 'Per 90 Minutes_G-PK', 'Per 90 Minutes_G+A-PK']


C:\Users\natmaw\AppData\Local\Temp\ipykernel_9836\805202972.py:46: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table), header=[0, 1])[0]


In [7]:
# ══════════════════════════════════════════════════════════════
# MAIN SCRAPING LOOP — only runs if test passed
# ══════════════════════════════════════════════════════════════
if RUN_FULL_SCRAPE:
    print("\n" + "=" * 60)
    print("FULL SCRAPE: Starting...")
    print(f"Leagues: {list(LEAGUES.keys())}")
    print(f"Seasons: {SEASONS}")
    print(f"Stat types: {list(STAT_TYPES.keys())}")
    total_pages = len(LEAGUES) * len(SEASONS) * len(STAT_TYPES)
    print(f"Total pages to scrape: {total_pages} (~{total_pages * 8 // 60} min)")
    print("=" * 60)

    time.sleep(8)  # Wait after the test request

    for league_key, league_info in LEAGUES.items():
        for season in SEASONS:
            print(f"\n{'━' * 60}")
            print(f"  {league_key.upper()} — {season}")
            print(f"{'━' * 60}")

            all_stats = {}

            for stat_type in STAT_TYPES:
                print(f"\n  [{stat_type}]")
                url = get_url(league_key, season, stat_type)
                df = scrape_table(url)

                if df is not None:
                    for col in df.columns:
                        try:
                            df[col] = pd.to_numeric(df[col])
                        except (ValueError, TypeError):
                            pass
                    all_stats[stat_type] = df
                else:
                    print(f"    SKIPPED")

                time.sleep(8)

            # Save CSV files for this league-season
            for stat_type, df_save in all_stats.items():
                df_save = df_save.copy()
                df_save["league"] = league_key
                df_save["season"] = season
                filename = f"{league_key}_{season}_{stat_type}.csv"
                filepath = os.path.join(OUTPUT_DIR, filename)
                df_save.to_csv(filepath, index=False, encoding="utf-8")
                print(f"    ✓ Saved: {filename} ({len(df_save)} rows)")

            print(f"\n  ✓ ALL SAVED for {league_key} {season}")

    # Summary
    print("\n" + "=" * 60)
    print("SCRAPE COMPLETE!")
    print("=" * 60)
    print(f"\nFiles saved to {OUTPUT_DIR}/:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        if f.endswith(".csv"):
            size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
            print(f"  {f} ({size:.0f} KB)")

    print(f"""
To load a file:
  df = pd.read_csv('data/raw/premier_league_2023-2024_standard.csv')
  print(df.head())
""")


FULL SCRAPE: Starting...
Leagues: ['premier_league', 'la_liga', 'bundesliga']
Seasons: ['2021-2022', '2022-2023', '2023-2024', '2024-2025']
Stat types: ['standard', 'shooting', 'passing', 'passing_types', 'goal_shot_creation', 'defense', 'possession', 'playing_time', 'misc']
Total pages to scrape: 108 (~14 min)


KeyboardInterrupt: 